# 📓 Caderno de Exercícios — Aula 03: Limpeza Numérica, Regras Matemáticas e Outliers
**Capacitação de Python para Ciência de Dados — Case Campe Supply**

> 💡 **O Coração Financeiro da Base:** Agora vamos tratar `faturamento`, `margem_de_lucro` e `lucro`. Lembre-se da equação fundamental do negócio:
> $$\text{Lucro} = \text{Faturamento} \times \text{Margem de Lucro}$$

In [ ]:
import pandas as pd
import numpy as np
from difflib import get_close_matches

# === Checkpoint Inicial (Trazendo os dados limpos das Aulas 01 e 02) ===
df = pd.read_excel('Campe_Supply_Corrompido.xlsx', sheet_name='Campe Supply')
df = df.drop(columns=['UNNAMED: 0', 'coluna_inutil_0', 'coluna_inutil_1'])
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('ã', 'a')
df = df.drop_duplicates().reset_index(drop=True)

categorias_oficiais = {
    'setor': ['Congelados', 'Latícinio', 'Bebidas', 'Carnes'],
    'produto': ['Queijo', 'Sorvete', 'Frango', 'Refrigerantes', 'Cervejas', 'Bovina', 'Leite', 'Lasanha', 'Pizza', 'Suína', 'Sucos', 'Coalhada'],
    'vendedor': ['Vanessa', 'Amaral', 'Leo', 'Daniele', 'Jonas', 'Evandro'],
    'regiao': ['Sudeste', 'Nordeste', 'Sul'],
    'estado': ['Rio de Janeiro', 'Minas Gerais', 'Bahia', 'Rio Grande do Norte', 'São Paulo', 'Sergipe', 'Rio Grande do Sul', 'Paraná', 'Santa Catarina'],
    'cliente': ['Atacarejo', 'Varejo', 'Atacado']
}
def corrigir_palavra(valor, lista_oficial):
    if pd.isna(valor): return np.nan
    match = get_close_matches(str(valor).strip(), lista_oficial, n=1, cutoff=0.6)
    return match[0] if match else np.nan

for col, lista_valida in categorias_oficiais.items():
    df[col] = df[col].apply(lambda x: corrigir_palavra(x, lista_valida))

mapa_produto_setor = {'Queijo': 'Latícinio', 'Leite': 'Latícinio', 'Coalhada': 'Latícinio', 'Sorvete': 'Congelados', 'Lasanha': 'Congelados', 'Pizza': 'Congelados', 'Frango': 'Carnes', 'Bovina': 'Carnes', 'Suína': 'Carnes', 'Refrigerantes': 'Bebidas', 'Cervejas': 'Bebidas', 'Sucos': 'Bebidas'}
mapa_estado_regiao = {'Rio de Janeiro': 'Sudeste', 'Minas Gerais': 'Sudeste', 'São Paulo': 'Sudeste', 'Bahia': 'Nordeste', 'Rio Grande do Norte': 'Nordeste', 'Sergipe': 'Nordeste', 'Rio Grande do Sul': 'Sul', 'Paraná': 'Sul', 'Santa Catarina': 'Sul'}
df['setor'] = df['setor'].fillna(df['produto'].map(mapa_produto_setor))
df['regiao'] = df['regiao'].fillna(df['estado'].map(mapa_estado_regiao))
print('Dados prontos para a Aula 03:', df.shape)

## 🎯 Missão 1: Investigando e Convertendo as Colunas Numéricas
As colunas `faturamento`, `margem_de_lucro` e `lucro` estão como `object` porque contêm erros de digitação com letras no meio dos números (ex: `"60b.72"`, `"0.a6"`).

1. Filtre ou exiba alguns desses valores corrompidos para ver como eles são.
2. Converta as 3 colunas para tipo numérico (`float`) usando `pd.to_numeric(..., errors='coerce')` (isso transformará os textos inválidos em `NaN` para podermos recalculá-los no próximo passo).
3. Rode um `df[['faturamento', 'margem_de_lucro', 'lucro']].describe()` e observe os valores mínimos e máximos!

In [ ]:
# 1. Converta as 3 colunas financeiras para numérico e analise o .describe()
# Seu código aqui:


## 🎯 Missão 2: Caçando e Corrigindo os 5 Outliers / Erros Matemáticos
Ao olhar o `.describe()`, você viu coisas absurdas como um **lucro máximo de R$ 3.260.000,00** e um **faturamento de R$ 8.581,60**!

1. Crie um filtro para encontrar todas as linhas onde a conta `faturamento * margem_de_lucro` difere de `lucro` em mais de R$ 0,01.
2. Você vai encontrar exatamente **5 linhas** com erro:
   * Em 2 linhas, o `lucro` está maior que o próprio `faturamento` (milhões). Corrija recalculando `lucro = faturamento * margem_de_lucro`.
   * Em 2 linhas, a `margem_de_lucro` está `0.00`, mas existe lucro positivo. Corrija recalculando `margem_de_lucro = lucro / faturamento`.
   * Em 1 linha, o `faturamento` foi multiplicado por 10 (`8581.60`), mas a margem (`0.06`) e o lucro (`51.4896`) estão certos. Corrija recalculando `faturamento = lucro / margem_de_lucro`.

In [ ]:
# 2. Identifique as 5 linhas com erro matemático e corrija cada caso usando a fórmula do negócio
# Seu código aqui:


## 🎯 Missão 3: Recuperando Nulos Numéricos com Matemática
Agora que não há mais outliers mentirosos, você tem mais de 100 células `NaN` nas 3 colunas numéricas. Porém, na maioria dessas linhas, **2 das 3 colunas estão preenchidas**!

Use a equação $\text{Lucro} = \text{Faturamento} \times \text{Margem}$ para preencher os `NaN` de:
* `lucro` (usando `faturamento * margem_de_lucro`)
* `faturamento` (usando `lucro / margem_de_lucro`)
* `margem_de_lucro` (usando `lucro / faturamento`)

Depois, arredonde as 3 colunas para 2 casas decimais (`.round(2)`).
* ✅ **Checkpoint de Validação:** Os nulos de `faturamento` devem cair de **115 para 20**, os de `margem_de_lucro` de **106 para 16**, e os de `lucro` de **116 para 13**!

In [ ]:
# 3. Recupere os valores nulos das 3 colunas financeiras e arredonde para 2 casas decimais
# Seu código aqui:


## 🎯 Missão 4: A Segunda Onda de Duplicatas e Engenharia de Datas
Agora que corrigimos textos (`"Vacessa" -> "Vanessa"`) e números (`"60b.72" -> 604.72`), linhas que eram idênticas — mas estavam disfarçadas por erros de digitação — ficaram iguais!

1. Verifique quantas novas duplicatas surgiram com `df.duplicated().sum()` e remova-as com `df.drop_duplicates()`.
2. Crie uma coluna `mes` extraindo o mês de `data_da_venda` (`df['data_da_venda'].dt.month`).

* ✅ **Checkpoint Final da Limpeza:** Sua base limpa terá exatamente **1.541 linhas únicas**!

In [ ]:
# 4. Remova as novas duplicatas reveladas, crie a coluna 'mes' e salve o Excel limpo se desejar
# Seu código aqui:
